# Imports

In [12]:
%cd ..

c:\Users\abhay\Desktop\abhay


In [13]:
import gymnasium as gym
from fnn import FNN
import torch
from torch import Tensor, nn, optim
from torch.nn.functional import mse_loss
from torch.nn.utils import clip_grad_norm_
from utils import device, get_ema, get_ema_and_emv, polyak_update

# Actor

In [10]:
class Actor(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, hidden_size: int | None = None) -> None:
        super().__init__()
        self._state_dim = state_dim
        self._action_dim = action_dim
        if hidden_size is None:
            hidden_size = state_dim + action_dim
        self._net = FNN(
            input_size = state_dim,
            hidden_size = hidden_size,
            num_hidden_layers = 4,
            output_size = action_dim,
        ).to(device)


    def forward(self, state: Tensor) -> Tensor:
        return self._net(state)

# Critic

In [11]:
class Critic(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, hidden_size: int | None = None) -> None:
        super().__init__()
        self._state_dim = state_dim
        self._action_dim = action_dim
        if hidden_size is None:
            hidden_size = state_dim + action_dim
        self._net = FNN(
            input_size = state_dim + action_dim,
            hidden_size = hidden_size,
            num_hidden_layers = 4,
            output_size = 1,
        ).to(device)


    def forward(self, state: Tensor, action: Tensor) -> Tensor:
        x = torch.cat([state, action], dim = -1)
        return self._net(x).squeeze(-1)

# Environment Setup

In [15]:
env = gym.make("HalfCheetah-v5")
state_dim = int(env.observation_space.shape[0])
action_dim = int(env.action_space.shape[0])